# Full Pipeline: Invoice Normalization

This notebook runs the complete end-to-end normalization pipeline on a sample invoice. The artifact it produces is deterministic — you can replay it with `paxman.replay()` (see notebook 07).

> **Order:** Run this notebook first, then open 07-replay to see determinism in action.

In [ ]:
from decimal import Decimal
from pathlib import Path
from pydantic import BaseModel, Field

import paxman
import paxman.contract.adapters.pydantic  # noqa: F401
import paxman.contract.adapters.dict_dsl  # noqa: F401

In [ ]:
INVOICE_PATH = Path("..") / "data" / "invoices" / "sample_invoice.txt"
invoice_text = INVOICE_PATH.read_text()
print(invoice_text)

In [ ]:
class Invoice(BaseModel):
    supplier: str
    invoice_number: str
    issue_date: str
    subtotal: Decimal
    currency: str = Field(default="MYR")
    total: Decimal


result = paxman.normalize(
    invoice_text, Invoice,
    policy=paxman.Policy(log_raw_input=False),
    budget=paxman.Budget(max_total_ms=2000),
)

print(f"Status: {result.status.name}")
print(f"Hash:   {result.replay_hash}")
print(f"Data:   {result.normalized_data}")

In [ ]:
for field_name, ev in result.evidence.items():
    print(f"--- {field_name} ---")
    for candidate in ev.evidence:
        print(f"  Value: {candidate.value!r}")
        print(f"  Confidence: {candidate.confidence.name}")
        print(f"  Source: {candidate.evidence_ref}")
    print()

In [ ]:
run2 = paxman.normalize(invoice_text, Invoice, policy=paxman.Policy(log_raw_input=False))
print(f"Original hash: {result.replay_hash}")
print(f"Second hash:   {run2.replay_hash}")
print(f"Match: {result.replay_hash == run2.replay_hash}")

## What happened

1. **Contract adaptation:** The Pydantic `Invoice` model was adapted to a `CanonicalContract` with 6 fields.
2. **Planning:** The planner created one `FieldPlan` per field.
3. **Execution:** The executor ran each plan, collecting evidence (candidates with confidence).
4. **Reconciliation:** The reconciler assigned final confidence and resolved conflicts.
5. **Artifact:** The artifact was serialized with a SHA-256 replay hash.

The artifact is deterministic — re-running with the same input + contract produces the same hash.

## Try it yourself

- Modify the input invoice and re-run. The hash should change.
- Add a line item field (ARRAY of objects) to the contract and see how the planner handles nested fields.
- Try a different contract format (Dict DSL instead of Pydantic) and verify the same artifact shape.
- Open notebook 07 to learn how `paxman.replay()` rehydrates artifacts without re-execution.